# Auditoria da Camada Bronze — Credit Risk Intelligence Platform

Este notebook implementa a **camada de auditoria operacional** da arquitetura Medalhão para as tabelas Bronze do catálogo `credit_risk`, seguindo boas práticas corporativas de Data Engineering, observabilidade, rastreabilidade e Data Governance.

### Escopo da Auditoria

| Categoria | Campos | Fonte |
|---|---|---|
| **Ingestão** | 20 | `_source_file`, `_ingestion_timestamp`, contexto Databricks |
| **Estatísticas de Carga** | 8 | `DESCRIBE DETAIL`, `df.count()` |
| **Controle de Execução** | 9 | `datetime`, try/except |
| **Delta Lake** | 8 | `DESCRIBE HISTORY` |
| **Qualidade** | 7 | `metadata_catalog_columns` + regras |
| **Governança** | 8 | Defaults de governança |
| **Lineage** | 8 | Mapeamento CSV → Delta |
| **Observabilidade** | 5 | Tendências, taxa de sucesso, crescimento |

### Tabela de Saída
- `credit_risk.bronze.audit_ingestion` — Delta Lake, append mode, versionada, consultas históricas

In [0]:
# ============================================================================
# Imports, constantes e captura de contexto de execução
# ============================================================================
from datetime import datetime, timezone
from pyspark.sql import Row
import uuid
import json
import logging

# Logging corporativo
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("audit_bronze")

# Constantes
CATALOG = "credit_risk"
SCHEMA_NAME = "bronze"
NOTEBOOK_NAME = "03_auditoria_bronze"
NOTEBOOK_PATH = "/Users/abraaojose.100@gmail.com/Projeto_classificação/Projeto_Credit_Risk/03_auditoria_bronze"
AUDIT_TABLE = f"{CATALOG}.{SCHEMA_NAME}.audit_ingestion"

# Contexto de execução (auto-extraído)
AUDIT_ID = str(uuid.uuid4())
EXECUTION_START = datetime.now(timezone.utc)
USER_NAME = spark.sql("SELECT current_user()").collect()[0][0]

# Tabelas Bronze para auditar (exclui tabelas de metadata/audit)
tabelas_bronze = [
    "application_train", "application_test", "bureau", "bureau_balance",
    "credit_card_balance", "installments_payments", "pos_cash_balance",
    "previous_application"
]

logger.info(f"Auditoria Bronze iniciada | ID: {AUDIT_ID} | Usuário: {USER_NAME}")
logger.info(f"Tabelas a auditar: {len(tabelas_bronze)}")

# Helper: acessa campos de DESCRIBE DETAIL testando camelCase e snake_case
def safe_get(d, *keys):
    """Tenta múltiplas chaves e retorna o primeiro valor encontrado."""
    for k in keys:
        if k in d:
            return d[k]
    return None

## 2. Auditoria de Ingestão e Estatísticas de Carga

Extrai automaticamente para cada tabela Bronze:
- **Ingestão**: `source_file`, `source_path`, `ingestion_timestamp` (das colunas `_source_file` e `_ingestion_timestamp`)
- **Estatísticas**: `record_count`, `column_count`, `file_count`, `table_size_bytes/mb/gb`, `avg_record_size`, `schema_version` (via `DESCRIBE DETAIL` e `df.count()`)

In [0]:
# ============================================================================
# Auditoria de Ingestão + Estatísticas de Carga
# ============================================================================
audit_data = []

for tabela in tabelas_bronze:
    full_table = f"{CATALOG}.{SCHEMA_NAME}.{tabela}"
    logger.info(f"Processando: {tabela}")

    df = spark.table(full_table)

    # --- Ingestion metadata (das colunas técnicas de ingestão) ---
    source_file = ""
    source_path = ""
    ingestion_ts = ""

    if "_source_file" in df.columns:
        r = df.select("_source_file").distinct().limit(1).collect()
        if r:
            source_file = r[0][0] or ""
            source_path = source_file.rsplit("/", 1)[0] if "/" in source_file else ""

    if "_ingestion_timestamp" in df.columns:
        r = df.agg({"_ingestion_timestamp": "max"}).collect()
        if r and r[0][0]:
            ingestion_ts = str(r[0][0])

    # --- DESCRIBE DETAIL: estatísticas da tabela ---
    detail = spark.sql(f"DESCRIBE DETAIL {full_table}").collect()[0].asDict()

    file_count = int(safe_get(detail, "numFiles", "num_files") or 0)
    size_bytes = int(safe_get(detail, "sizeInBytes", "size_in_bytes") or 0)
    size_mb = round(size_bytes / (1024 * 1024), 2)
    table_type = safe_get(detail, "tableType", "table_type") or ""
    ds_format = safe_get(detail, "dataSourceFormat", "data_source_format") or ""
    created_at_val = safe_get(detail, "createdAt", "created_at")
    last_modified_val = safe_get(detail, "lastModified", "last_modified")
    min_reader = safe_get(detail, "minReaderVersion", "min_reader_version") or 1
    min_writer = safe_get(detail, "minWriterVersion", "min_writer_version") or 2
    partition_cols = safe_get(detail, "partitionColumns", "partition_columns") or []

    # --- Load statistics ---
    record_count = df.count()
    column_count = len(df.columns)
    avg_record_size = round(size_bytes / max(record_count, 1), 2)
    schema_version = f"reader={min_reader},writer={min_writer}"

    audit_data.append({
        "table_name": tabela,
        "source_system": "Home Credit Default Risk (Kaggle)",
        "source_file": source_file,
        "source_path": source_path,
        "source_layer": "Volume",
        "target_layer": "Bronze",
        "ingestion_timestamp": ingestion_ts,
        "record_count": record_count,
        "column_count": column_count,
        "file_count": file_count,
        "table_size_bytes": size_bytes,
        "table_size_mb": size_mb,
        "table_size_gb": round(size_mb / 1024, 4),
        "avg_record_size": avg_record_size,
        "schema_version": schema_version,
        "table_type": table_type,
        "data_source_format": ds_format,
        "table_created_at": str(created_at_val) if created_at_val else "",
        "table_last_modified": str(last_modified_val) if last_modified_val else "",
        "partition_columns": ",".join(partition_cols),
    })

    logger.info(f"  {tabela}: {record_count} registros | {file_count} arquivos | {size_mb} MB")

logger.info(f"Ingestão + estatísticas: {len(audit_data)} tabelas processadas")

## 3. Auditoria Delta Lake

Captura metadados do Delta Lake via `DESCRIBE HISTORY`:
- `delta_version`, `operation`, `operation_parameters` (JSON), `operation_metrics` (JSON)
- `last_commit_timestamp`, `user_metadata`
- Contagem de operações `OPTIMIZE` e `VACUUM` no histórico

In [0]:
# ============================================================================
# Auditoria Delta Lake (DESCRIBE HISTORY)
# ============================================================================
for entry in audit_data:
    tabela = entry["table_name"]
    full_table = f"{CATALOG}.{SCHEMA_NAME}.{tabela}"

    # Última operação Delta
    history = spark.sql(f"DESCRIBE HISTORY {full_table} LIMIT 1").collect()

    if history:
        h = history[0].asDict()
        entry["delta_version"] = int(h.get("version", 0))
        entry["last_commit_timestamp"] = str(h.get("timestamp", ""))
        entry["operation"] = h.get("operation", "")
        entry["operation_parameters"] = json.dumps(h.get("operationParameters", {}), default=str)
        entry["operation_metrics"] = json.dumps(h.get("operationMetrics", {}), default=str)
        entry["user_metadata"] = json.dumps(h.get("userMetadata", {}), default=str)
    else:
        entry["delta_version"] = 0
        entry["last_commit_timestamp"] = ""
        entry["operation"] = ""
        entry["operation_parameters"] = "{}"
        entry["operation_metrics"] = "{}"
        entry["user_metadata"] = "{}"

    # Contagem de OPTIMIZE e VACUUM no histórico completo
    full_history = spark.sql(f"DESCRIBE HISTORY {full_table}").collect()
    optimize_count = sum(1 for r in full_history if r["operation"] == "OPTIMIZE")
    vacuum_count = sum(1 for r in full_history if r["operation"] == "VACUUM")
    entry["optimize_history"] = optimize_count
    entry["vacuum_history"] = vacuum_count

    logger.info(f"  {tabela}: Delta v{entry['delta_version']} | op={entry['operation']} | optimize={optimize_count}")

logger.info(f"Delta Lake audit: {len(audit_data)} tabelas")

## 4. Auditoria de Qualidade e Governança

**Qualidade**: lê `metadata_catalog_columns` (criada no notebook `02_Metadados_bronze`) para agregações de `null_count`, `null_percentage` por tabela. Calcula `quality_score` e `quality_status`.

**Governança**: defaults corporativos (domain, data_classification, sensitivity_level, lgpd_flag). Campos manuais preparados (data_owner, data_steward, retention_policy).

In [0]:
# ============================================================================
# Auditoria de Qualidade e Governança
# ============================================================================

# Tenta ler quality stats da tabela metadata_catalog_columns
quality_stats = {}
try:
    if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_NAME}.metadata_catalog_columns"):
        quality_df = spark.sql(f"""
            SELECT table_name,
                   SUM(null_count) as total_nulls,
                   AVG(null_percentage) as avg_null_pct,
                   SUM(distinct_count) as total_distinct
            FROM {CATALOG}.{SCHEMA_NAME}.metadata_catalog_columns
            GROUP BY table_name
        """).collect()
        for row in quality_df:
            quality_stats[row["table_name"]] = {
                "total_nulls": int(row["total_nulls"] or 0),
                "avg_null_pct": float(row["avg_null_pct"] or 0.0),
                "total_distinct": int(row["total_distinct"] or 0),
            }
        logger.info(f"Quality stats lidas de metadata_catalog_columns: {len(quality_stats)} tabelas")
except Exception as e:
    logger.warning(f"metadata_catalog_columns não disponível: {e}")

for entry in audit_data:
    tabela = entry["table_name"]
    qs = quality_stats.get(tabela, {})

    avg_null_pct = qs.get("avg_null_pct", 0.0)
    quality_score = round(100 - avg_null_pct, 2)

    if avg_null_pct < 5:
        quality_status = "passed"
    elif avg_null_pct < 20:
        quality_status = "warning"
    else:
        quality_status = "failed"

    entry["null_count"] = qs.get("total_nulls", 0)
    entry["null_percentage"] = avg_null_pct
    entry["duplicate_count"] = 0
    entry["duplicate_percentage"] = 0.0
    entry["quality_score"] = quality_score
    entry["quality_status"] = quality_status
    entry["validation_status"] = "pending"

    # Governança (defaults corporativos)
    entry["data_owner"] = ""
    entry["data_steward"] = ""
    entry["domain"] = "Credit Risk"
    entry["data_classification"] = "Confidential"
    entry["sensitivity_level"] = "High"
    entry["retention_policy"] = ""
    entry["regulatory_requirement"] = ""
    entry["lgpd_flag"] = True

    logger.info(f"  {tabela}: quality_score={quality_score} | status={quality_status}")

## 5. Auditoria de Lineage

Mapeamento estatico fonte → destino (Volume CSV → Bronze Delta) para rastreabilidade.

In [0]:
# ============================================================================
# Auditoria de Lineage: CSV (Volume) → Delta (Bronze)
# ============================================================================
VOLUME_BASE = "/Volumes/credit_risk/bronze/volume/home-credit-default-risk/"
CSV_TO_TABLE = {
    "application_train.csv": "application_train",
    "application_test.csv": "application_test",
    "bureau.csv": "bureau",
    "bureau_balance.csv": "bureau_balance",
    "credit_card_balance.csv": "credit_card_balance",
    "installments_payments.csv": "installments_payments",
    "POS_CASH_balance.csv": "pos_cash_balance",
    "previous_application.csv": "previous_application",
}
TABLE_TO_CSV = {v: k for k, v in CSV_TO_TABLE.items()}

for entry in audit_data:
    tabela = entry["table_name"]
    csv_file = TABLE_TO_CSV.get(tabela, "")

    entry["source_catalog"] = ""
    entry["source_schema"] = ""
    entry["source_table"] = csv_file
    entry["target_catalog"] = CATALOG
    entry["target_schema"] = SCHEMA_NAME
    entry["target_table"] = tabela
    entry["transformation_name"] = "CSV → Delta (Auto Loader)"
    entry["transformation_type"] = "ingestion"

logger.info(f"Lineage: {len(audit_data)} mapeamentos fonte → destino")

## 6. Observabilidade e Indicadores Operacionais

Indicadores calculados a partir de execuções anteriores da auditoria:
- `table_growth_pct`: crescimento da tabela vs última auditoria
- `ingestion_trend`: increasing / stable / initial
- `success_rate`: taxa de sucesso historica
- `avg_processing_time`: tempo médio de processamento
- `volume_processed_mb`: volume processado

In [0]:
# ============================================================================
# Observabilidade: indicadores operacionais (de auditorias anteriores)
# ============================================================================
prev_stats = {}
try:
    if spark.catalog.tableExists(AUDIT_TABLE):
        prev_df = spark.sql(f"""
            SELECT table_name,
                   MAX(table_size_mb) as prev_size,
                   MAX(record_count) as prev_count,
                   AVG(duration_seconds) as avg_duration,
                   SUM(CASE WHEN success_flag = true THEN 1 ELSE 0 END) as success_count,
                   COUNT(*) as total_runs
            FROM {AUDIT_TABLE}
            GROUP BY table_name
        """).collect()
        for row in prev_df:
            prev_stats[row["table_name"]] = {
                "prev_size": float(row["prev_size"] or 0),
                "prev_count": int(row["prev_count"] or 0),
                "avg_duration": float(row["avg_duration"] or 0),
                "success_count": int(row["success_count"] or 0),
                "total_runs": int(row["total_runs"] or 0),
            }
        logger.info(f"Auditoria anterior encontrada: {len(prev_stats)} tabelas")
except Exception as e:
    logger.warning(f"Não foi possível ler auditoria anterior: {e}")

for entry in audit_data:
    tabela = entry["table_name"]
    prev = prev_stats.get(tabela, {})

    # Crescimento da tabela
    prev_size = prev.get("prev_size", 0)
    if prev_size > 0:
        entry["table_growth_pct"] = round((entry["table_size_mb"] - prev_size) / prev_size * 100, 2)
    else:
        entry["table_growth_pct"] = 0.0

    # Tendência de ingestão
    prev_count = prev.get("prev_count", 0)
    if prev_count > 0:
        entry["ingestion_trend"] = "increasing" if entry["record_count"] > prev_count else "stable"
    else:
        entry["ingestion_trend"] = "initial"

    # Taxa de sucesso histórica
    total_runs = prev.get("total_runs", 0)
    success_count = prev.get("success_count", 0)
    entry["success_rate"] = round(success_count / total_runs * 100, 2) if total_runs > 0 else 100.0

    # Tempo médio de processamento
    entry["avg_processing_time"] = prev.get("avg_duration", 0.0)

    # Volume processado
    entry["volume_processed_mb"] = entry["table_size_mb"]

logger.info(f"Observabilidade: {len(audit_data)} indicadores calculados")

## 7. Consolidação e Persistência — `credit_risk.bronze.audit_ingestion`

Consolida todos os metadados em um único DataFrame com **74 campos** e persiste como tabela Delta.

**Modo append**: cada execução adiciona 8 registros (um por tabela), permitindo consultas históricas e monitoramento operacional ao longo do tempo.

In [0]:
# ============================================================================
# Controle de Execução + Consolidação e Persistência
# ============================================================================
EXECUTION_END = datetime.now(timezone.utc)
duration_seconds = (EXECUTION_END - EXECUTION_START).total_seconds()
duration_minutes = round(duration_seconds / 60, 2)

EXECUTION_STATUS = "success"
SUCCESS_FLAG = True
ERROR_FLAG = False
ERROR_MESSAGE = ""
WARNING_MESSAGE = ""

logger.info(f"Consolidando auditoria... | Duração: {duration_seconds:.1f}s")

# Construir DataFrame final com todos os 74 campos
catalogo_audit = []

for entry in audit_data:
    record = {
        # --- Auditoria de Ingestão (21) ---
        "audit_id": AUDIT_ID,
        "audit_timestamp": EXECUTION_START.isoformat(),
        "table_name": entry["table_name"],
        "source_system": entry["source_system"],
        "source_file": entry["source_file"],
        "source_path": entry["source_path"],
        "source_layer": entry["source_layer"],
        "target_layer": entry["target_layer"],
        "ingestion_timestamp": entry["ingestion_timestamp"],
        "processing_timestamp": EXECUTION_START.isoformat(),
        "load_timestamp": EXECUTION_END.isoformat(),
        "batch_id": "",
        "execution_id": AUDIT_ID,
        "run_id": "",
        "pipeline_id": "",
        "notebook_name": NOTEBOOK_NAME,
        "notebook_path": NOTEBOOK_PATH,
        "job_id": "",
        "cluster_id": "",
        "workspace_name": "",
        "user_name": USER_NAME,
        # --- Estatísticas de Carga (8) ---
        "record_count": entry["record_count"],
        "column_count": entry["column_count"],
        "file_count": entry["file_count"],
        "table_size_bytes": entry["table_size_bytes"],
        "table_size_mb": entry["table_size_mb"],
        "table_size_gb": entry["table_size_gb"],
        "avg_record_size": entry["avg_record_size"],
        "schema_version": entry["schema_version"],
        # --- Controle de Execução (9) ---
        "start_time": EXECUTION_START.isoformat(),
        "end_time": EXECUTION_END.isoformat(),
        "duration_seconds": round(duration_seconds, 2),
        "duration_minutes": duration_minutes,
        "execution_status": EXECUTION_STATUS,
        "success_flag": SUCCESS_FLAG,
        "error_flag": ERROR_FLAG,
        "error_message": ERROR_MESSAGE,
        "warning_message": WARNING_MESSAGE,
        # --- Auditoria Delta Lake (8) ---
        "delta_version": entry["delta_version"],
        "operation": entry["operation"],
        "operation_parameters": entry["operation_parameters"],
        "operation_metrics": entry["operation_metrics"],
        "user_metadata": entry["user_metadata"],
        "last_commit_timestamp": entry["last_commit_timestamp"],
        "optimize_history": entry["optimize_history"],
        "vacuum_history": entry["vacuum_history"],
        # --- Auditoria de Qualidade (7) ---
        "null_count": entry["null_count"],
        "null_percentage": entry["null_percentage"],
        "duplicate_count": entry["duplicate_count"],
        "duplicate_percentage": entry["duplicate_percentage"],
        "quality_score": entry["quality_score"],
        "quality_status": entry["quality_status"],
        "validation_status": entry["validation_status"],
        # --- Auditoria de Governança (8) ---
        "data_owner": entry["data_owner"],
        "data_steward": entry["data_steward"],
        "domain": entry["domain"],
        "data_classification": entry["data_classification"],
        "sensitivity_level": entry["sensitivity_level"],
        "retention_policy": entry["retention_policy"],
        "regulatory_requirement": entry["regulatory_requirement"],
        "lgpd_flag": entry["lgpd_flag"],
        # --- Auditoria de Lineage (8) ---
        "source_catalog": entry["source_catalog"],
        "source_schema": entry["source_schema"],
        "source_table": entry["source_table"],
        "target_catalog": entry["target_catalog"],
        "target_schema": entry["target_schema"],
        "target_table": entry["target_table"],
        "transformation_name": entry["transformation_name"],
        "transformation_type": entry["transformation_type"],
        # --- Observabilidade (5) ---
        "volume_processed_mb": entry["volume_processed_mb"],
        "table_growth_pct": entry["table_growth_pct"],
        "ingestion_trend": entry["ingestion_trend"],
        "success_rate": entry["success_rate"],
        "avg_processing_time": entry["avg_processing_time"],
    }
    catalogo_audit.append(Row(**record))

df_audit = spark.createDataFrame(catalogo_audit)

# Persistir como Delta (append mode para histórico)
if spark.catalog.tableExists(AUDIT_TABLE):
    df_audit.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)
    logger.info(f"Registros appendados a {AUDIT_TABLE}")
else:
    df_audit.write.format("delta").mode("overwrite").saveAsTable(AUDIT_TABLE)
    logger.info(f"Tabela {AUDIT_TABLE} criada")

print(f"Tabela {AUDIT_TABLE} atualizada com sucesso!")
print(f"Registros adicionados: {df_audit.count()}")
print(f"Campos: {len(df_audit.columns)}")

## 8. Verificação e Estatísticas Finais

Exibe estatísticas consolidadas, classificação de maturidade e roadmap para evolução.

In [0]:
# ============================================================================
# Verificação e Estatísticas Finais
# ============================================================================
print("=" * 70)
print("AUDITORIA DA CAMADA BRONZE — CREDIT RISK INTELLIGENCE PLATFORM")
print("=" * 70)

# Total de registros historicos
total = spark.sql(f"SELECT COUNT(*) as total FROM {AUDIT_TABLE}").collect()[0]["total"]
print(f"\n📊 Total de registros de auditoria (histórico): {total}")

# Última execução
print(f"\n📊 Última execução (audit_id={AUDIT_ID[:8]}...):")
spark.sql(f"""
    SELECT table_name, record_count, table_size_mb, delta_version,
           quality_score, quality_status, duration_seconds, execution_status
    FROM {AUDIT_TABLE}
    WHERE audit_id = '{AUDIT_ID}'
    ORDER BY table_name
""").show(truncate=False)

# Resumo por tabela (histórico)
print(f"📊 Resumo de auditoria por tabela (histórico):")
spark.sql(f"""
    SELECT table_name,
           COUNT(*) as audit_runs,
           MAX(record_count) as max_records,
           MAX(table_size_mb) as max_size_mb,
           AVG(duration_seconds) as avg_duration_sec,
           SUM(CASE WHEN success_flag = true THEN 1 ELSE 0 END) as successes,
           COUNT(*) as total_runs
    FROM {AUDIT_TABLE}
    GROUP BY table_name
    ORDER BY table_name
""").show(truncate=False)

# Qualidade
print(f"📊 Status de qualidade (última execução):")
spark.sql(f"""
    SELECT table_name, quality_score, quality_status, null_percentage
    FROM {AUDIT_TABLE}
    WHERE audit_id = '{AUDIT_ID}'
    ORDER BY table_name
""").show(truncate=False)

# Lineage
print(f"📊 Lineage (última execução):")
spark.sql(f"""
    SELECT source_table, target_table, transformation_type
    FROM {AUDIT_TABLE}
    WHERE audit_id = '{AUDIT_ID}'
    ORDER BY target_table
""").show(truncate=False)

# Delta Lake audit
print(f"📊 Delta Lake (última execução):")
spark.sql(f"""
    SELECT table_name, delta_version, operation, optimize_history, vacuum_history
    FROM {AUDIT_TABLE}
    WHERE audit_id = '{AUDIT_ID}'
    ORDER BY table_name
""").show(truncate=False)

# Observabilidade
print(f"📊 Observabilidade (última execução):")
spark.sql(f"""
    SELECT table_name, volume_processed_mb, table_growth_pct, ingestion_trend,
           success_rate, avg_processing_time
    FROM {AUDIT_TABLE}
    WHERE audit_id = '{AUDIT_ID}'
    ORDER BY table_name
""").show(truncate=False)

# Classificação de maturidade
print(f"\n📋 CLASSIFICAÇÃO DE MATURIDADE DA AUDITORIA:")
print(f"   ✅ IMPLEMENTADO (auto-extraído):")
print(f"      Ingestão: source_file, source_path, ingestion_timestamp, user_name, notebook_name")
print(f"      Estatísticas: record_count, file_count, table_size_mb, avg_record_size, schema_version")
print(f"      Execução: start_time, end_time, duration_seconds, execution_status")
print(f"      Delta Lake: delta_version, operation, operation_metrics, optimize_history, vacuum_history")
print(f"      Qualidade: null_count, null_percentage, quality_score, quality_status")
print(f"      Governança: domain, data_classification, sensitivity_level, lgpd_flag")
print(f"      Lineage: source_table, target_table, transformation_type")
print(f"      Observabilidade: table_growth_pct, ingestion_trend, success_rate, avg_processing_time")
print(f"   📝 PENDENTE (preencher via integração):")
print(f"      batch_id, job_id, run_id (via Databricks Jobs)")
print(f"      data_owner, data_steward, retention_policy (via governança de negócio)")
print(f"      duplicate_count, duplicate_percentage (via Data Quality framework)")
print(f"      validation_status (via Great Expectations / DQ rules)")
print(f"   🔮 ROADMAP DE EVOLUÇÃO:")
print(f"      1. Alertas automatizados (quality_score < threshold → notificação)")
print(f"      2. Dashboard operacional (Power BI / Lakeview)")
print(f"      3. Integração com Databricks Jobs (batch_id, run_id automáticos)")
print(f"      4. Data Quality framework (Great Expectations / DLT expectations)")
print(f"      5. Extensão para camadas Silver e Gold")
print(f"      6. Integração com MLOps (tracking de features e modelos)")
print(f"      7. Time travel queries para auditoria retroativa")
print(f"      8. Data lineage automatizado via Unity Catalog lineage APIs")